# Ejercicio 2 — Ascenso por máxima pendiente desde un $3^2$ (R)

**Objetivo.** Ajustar modelo de primer orden sobre $3^2$ y calcular la trayectoria
de ascenso por máxima pendiente.

**Factores:** pH ($A$: 5.5/6.5/7.5) y Temperatura ($B$: 28/32/36 °C)
**Respuesta:** Biomasa (g/L)

In [ ]:
library(dplyr)
library(ggplot2)
library(rsm)

df <- read.csv('../../datos/fermentacion-3k.csv')
print(df)

## 1. Verificación de curvatura

Ajustamos el modelo completo de segundo orden para comprobar si hay curvatura en la región
actual. Si los p-valores de los términos cuadráticos ($x_1^2$, $x_2^2$) son > 0.05, el
modelo de primer orden es adecuado y podemos calcular la dirección de ascenso.

> Si la curvatura fuera significativa, se augmentaría el diseño a un CCD en lugar de
> calcular la trayectoria de ascenso.

In [ ]:
modelo_2o <- lm(biomasa ~ x1 + x2 + I(x1^2) + I(x2^2) + x1:x2, data = df)
print(anova(modelo_2o))

## 2. Modelo de primer orden

In [ ]:
modelo_1o <- lm(biomasa ~ x1 + x2, data = df)
print(summary(modelo_1o))

b0 <- coef(modelo_1o)['(Intercept)']
b1 <- coef(modelo_1o)['x1']
b2 <- coef(modelo_1o)['x2']
cat(sprintf('β₀=%.3f  β₁=%.3f  β₂=%.3f\n', b0, b1, b2))

## 3. Trayectoria de ascenso

In [ ]:
grad <- c(b1, b2)
grad_norm <- grad / sqrt(sum(grad^2))
centro_real <- c(6.5, 32.0)
delta_real  <- c(1.0,  4.0)
paso_base   <- 0.3

traj <- do.call(rbind, lapply(0:5, function(s) {
  x_cod  <- grad_norm * s * paso_base
  x_real <- centro_real + x_cod * delta_real
  y_hat  <- b0 + b1 * x_cod[1] + b2 * x_cod[2]
  data.frame(paso=s, x1=round(x_cod[1],3), x2=round(x_cod[2],3),
             pH=round(x_real[1],2), T=round(x_real[2],2), y_hat=round(y_hat,2))
}))
print(traj)

## 4. Gráfico de contornos con trayectoria

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
grilla <- expand.grid(x1 = seq(-1.5, 2.5, by=0.05),
                      x2 = seq(-1.5, 2.5, by=0.05))
grilla$y_hat <- b0 + b1*grilla$x1 + b2*grilla$x2

ggplot(grilla, aes(x=x1, y=x2, z=y_hat)) +
  geom_contour_filled(bins=15) +
  geom_point(data=df, aes(x=x1, y=x2), inherit.aes=FALSE, color='black', size=3) +
  geom_path(data=traj, aes(x=x1, y=x2), inherit.aes=FALSE, color='red', linewidth=1.2) +
  geom_point(data=traj, aes(x=x1, y=x2), inherit.aes=FALSE,
             color='red', shape=21, fill='white', size=3) +
  labs(x=expression(x[1]*' (pH)'), y=expression(x[2]*' (Temperatura)'),
       title='Ascenso por máxima pendiente desde 3²', fill='Biomasa (g/L)') +
  theme_minimal(base_size=13)

## 5. Conclusión

- El primer paso en RSM es **verificar curvatura**: en este ejercicio los términos cuadráticos
  no son significativos, por lo que el modelo de primer orden es válido en la región actual.
- El gradiente $(\hat\beta_1, \hat\beta_2)$ del modelo de primer orden define la dirección
  de ascenso por máxima pendiente.
- La trayectoria se ejecuta experimentalmente; cuando la respuesta deja de crecer se centra
  un nuevo diseño (CCD o $3^2$) en ese punto para ajustar el modelo de segundo orden.